In [4]:
# -*- coding: utf-8 -*-
"""
深度学习系列之二复现（聚宽真实数据版）
GRU + IC损失：截面选股 + 时序择时统一框架

运行环境：聚宽(JoinQuant)研究环境（Notebook）。直接整段粘贴/上传运行即可。
若在本地运行，请设置 JQ_USER / JQ_PASS 使用 jqdatasdk。

数据获取方式（参考 DecompGRU 文档中的 read_data 思路：行情 + 估值 + 行业，按 code/time 合并、清洗、去重）：
  - 行情  ：get_price  -> open/close/low/high/avg/volume/money（前复权，avg 即 VWAP = money/volume）
  - 市值  ：get_valuation -> market_cap（仅用于股票池按市值筛选/描述，标签保持原始绝对收益，不做中性化/截面标准化）
  - 行业  ：get_industry -> sw_l1（可选，仅输出行业分布；研报模型不使用）
  - 指数  ：get_price(指数代码) 用于“直接推理择时”（中证全指 000985.XSHG）

与研报一致的设置：窗口=40日，标签=下一交易日起算的未来5日VWAP收益率（原始值），IC损失，
样本隔离（剔除末尾 N+1 天；训练/验证交界剔除 (N-1)/2 天），按交易日 batch，早停。

注意：股票池/样本量远小于研报（全A、2018-2026），结果仅用于流程验证。
"""
import warnings
warnings.filterwarnings("ignore")
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
try:
    from numpy.lib.stride_tricks import sliding_window_view
except ImportError:   # 聚宽 numpy 版本较旧(<1.20)，用 as_strided 手写等价实现
    from numpy.lib.stride_tricks import as_strided

    def sliding_window_view(a, window, axis=0):
        """仅支持 axis=0：(n, S) -> (n-window+1, S, window)，与新版 numpy 行为一致（只读视图）"""
        assert axis == 0 and a.ndim == 2
        a = np.ascontiguousarray(a)
        n, s = a.shape
        return as_strided(a, shape=(n - window + 1, s, window),
                          strides=(a.strides[0], a.strides[1], a.strides[0]), writeable=False)
import matplotlib
import matplotlib.pyplot as plt

# ---------------- 聚宽数据接口兼容 ----------------
try:
    from jqdata import *          # 研究环境
    _JQ = "research"
except Exception:
    import jqdatasdk as jq        # 本地：pip install jqdatasdk
    jq.auth(os.environ.get("JQ_USER", ""), os.environ.get("JQ_PASS", ""))
    from jqdatasdk import *
    _JQ = "sdk"

plt.rcParams["axes.unicode_minus"] = False
for _f in ["SimHei", "Noto Sans CJK SC", "WenQuanYi Zen Hei", "Heiti TC"]:
    try:
        matplotlib.font_manager.findfont(_f, fallback_to_default=False)
        plt.rcParams["font.sans-serif"] = [_f]
        break
    except Exception:
        continue

In [5]:
# =========================================================
# 配置
# =========================================================
CFG = dict(
    START="2024-01-01",          # 数据起始（含窗口预热）
    END="2026-10-20",
    UNIVERSE_INDEX="000300.XSHG",  # 股票池：该指数成分股（按END日成分，存在幸存者偏差）
    TIMING_INDEX="000985.XSHG",    # 直接推理择时标的：中证全指
    WINDOW=40, HORIZON=5,
    HIDDEN=16, LAYERS=2, LR=2e-3,
    MAX_EPOCH=20, PATIENCE=6,
    CHUNK=20,                    # 每次 get_price 的股票数（避免单次返回行数超限）
    MIN_LIST_DAYS=365,           # 剔除上市不满365天样本
    OUT_DIR="./outputs",
)
WINDOW, HORIZON = CFG["WINDOW"], CFG["HORIZON"]
os.makedirs(CFG["OUT_DIR"], exist_ok=True)
torch.manual_seed(42)
np.random.seed(42)

PRICE_FIELDS = ["open", "close", "low", "high", "avg", "volume", "money"]


# =========================================================
# 第一部分：聚宽真实数据读取与清洗（对应文档1的 read_data）
# =========================================================
def read_data():
    # 1) 股票池 + 上市日期
    codes = get_index_stocks(CFG["UNIVERSE_INDEX"], date=CFG["END"])
    info = get_all_securities(["stock"], date=CFG["END"])
    codes = [c for c in codes if c in info.index]
    list_date = pd.to_datetime(info.loc[codes, "start_date"])

    # 2) 行情（分块拉取，前复权，保留停牌日为NaN以便识别）
    frames = []
    for i in range(0, len(codes), CFG["CHUNK"]):
        sub = codes[i:i + CFG["CHUNK"]]
        d = get_price(sub, start_date=CFG["START"], end_date=CFG["END"], frequency="daily",
                      fields=PRICE_FIELDS, skip_paused=False, fq="pre", panel=False)
        frames.append(d)
        print(f"行情进度 {min(i + CFG['CHUNK'], len(codes))}/{len(codes)}")
    df_base = pd.concat(frames, ignore_index=True)
    df_base["time"] = pd.to_datetime(df_base["time"])

    # 3) 估值（市值）——与文档1一致取 market_cap，用于描述/可选筛选
    mc_list = []
    for dt in pd.to_datetime(df_base["time"].unique())[::20]:   # 抽样日期，降低调用量
        q = query(valuation.code, valuation.market_cap).filter(valuation.code.in_(codes))
        v = get_fundamentals(q, date=dt)
        v["time"] = dt
        mc_list.append(v)
    df_val = pd.concat(mc_list, ignore_index=True) if mc_list else pd.DataFrame()

    # 4) 行业（申万一级，仅描述用途）
    try:
        ind = get_industry(codes, date=CFG["END"])
        df_ind = pd.DataFrame({"code": list(ind.keys()),
                               "industry": [str(v.get("sw_l1", {}).get("industry_code", "")) for v in ind.values()]})
    except Exception:
        df_ind = pd.DataFrame(columns=["code", "industry"])

    # 5) 清洗：数值化、去重、排序（与文档1一致）
    for c in PRICE_FIELDS:
        df_base[c] = pd.to_numeric(df_base[c], errors="coerce")
    df_base = df_base.drop_duplicates(subset=["code", "time"]).sort_values(["code", "time"]).reset_index(drop=True)
    return df_base, df_val, df_ind, list_date


def to_panel(df, col):
    return df.pivot(index="time", columns="code", values=col).sort_index()


print("=" * 60, "\n第一步：从聚宽读取真实数据\n", "=" * 60)
df_base, df_val, df_ind, list_date = read_data()
panels = {c: to_panel(df_base, c) for c in PRICE_FIELDS}
dates = panels["close"].index
codes = list(panels["close"].columns)
n_days, n_stocks = panels["close"].shape
print(f"股票数 {n_stocks}，交易日数 {n_days}，区间 {dates[0].date()} ~ {dates[-1].date()}")
if len(df_ind):
    print("申万一级行业分布(前5)：", df_ind["industry"].value_counts().head(5).to_dict())

# 停牌日(volume==0 或 NaN)置为NaN，防止污染特征
halt = ~(panels["volume"] > 0)
for c in PRICE_FIELDS:
    panels[c] = panels[c].mask(halt)

# 研报特征：open/close/high/low/vwap/volume/amount，vwap 取聚宽 avg
RAW = {"open": panels["open"].values, "close": panels["close"].values,
       "high": panels["high"].values, "low": panels["low"].values,
       "vwap": panels["avg"].values, "volume": panels["volume"].values,
       "amount": panels["money"].values}
FEATURES = ["open", "close", "high", "low", "vwap", "volume", "amount"]

# 上市满365天掩码 (n_days, n_stocks)
ld = list_date.reindex(codes).values
listed_ok = dates.values[:, None] >= (ld[None, :] + np.timedelta64(CFG["MIN_LIST_DAYS"], "D"))

第一步：从聚宽读取真实数据
行情进度 20/300
行情进度 40/300
行情进度 60/300
行情进度 80/300
行情进度 100/300
行情进度 120/300
行情进度 140/300
行情进度 160/300
行情进度 180/300
行情进度 200/300
行情进度 220/300
行情进度 240/300
行情进度 260/300
行情进度 280/300
行情进度 300/300
股票数 300，交易日数 675，区间 2024-01-02 ~ 2026-10-20
申万一级行业分布(前5)： {'801080': 38, '801790': 26, '801780': 24, '801730': 21, '801150': 18}


In [6]:
df_base.head()

,time,code,open,close,low,high,avg,volume,money
0,2024-01-02,000001.XSHE,7.71,7.57,7.57,7.74,7.63,141003802.0,1.075742e+09
1,2024-01-03,000001.XSHE,7.55,7.56,7.52,7.57,7.54,89299757.0,6.736736e+08
2,2024-01-04,000001.XSHE,7.55,7.48,7.46,7.55,7.49,105195241.0,7.874701e+08
3,2024-01-05,000001.XSHE,7.48,7.62,7.45,7.76,7.64,242433037.0,1.852660e+09
4,2024-01-08,000001.XSHE,7.58,7.52,7.48,7.64,7.54,136474330.0,1.029007e+09


In [7]:
RAW

{'open': array([[7.71, 10.44, 25.07, ..., 141.5, 49.82, 53.02],
        [7.55, 10.15, 24.12, ..., 142.0, 47.54, 52.85],
        [7.55, 10.1, 24.04, ..., 138.75, 46.53, 52.5],
        ...,
        [11.41, 3.62, 27.15, ..., 233.33, 128.0, 105.0],
        [11.48, 3.59, 26.9, ..., 231.88, 125.86, 100.87],
        [11.62, 3.51, 27.13, ..., 238.78, 124.66, 98.99]]),
 'close': array([[7.57, 10.15, 24.21, ..., 142.0, 47.78, 52.96],
        [7.56, 10.13, 23.96, ..., 138.99, 46.53, 52.48],
        [7.48, 9.93, 23.59, ..., 136.82, 45.7, 51.88],
        ...,
        [11.5, 3.66, 27.1, ..., 233.88, 126.1, 102.66],
        [11.65, 3.5, 27.27, ..., 238.0, 124.11, 99.7],
        [11.44, 3.45, 27.56, ..., 253.23, 130.0, 101.98]]),
 'high': array([[7.74, 10.48, 25.08, ..., 144.5, 49.93, 53.27],
        [7.57, 10.23, 24.38, ..., 142.0, 48.08, 53.18],
        [7.55, 10.11, 24.04, ..., 138.75, 47.78, 52.7],
        ...,
        [11.63, 3.85, 27.48, ..., 242.99, 131.3, 105.0],
        [11.68, 3.65, 27.34, .

In [8]:
, df_ind, list_date 

('df_ind,', 'list_date')

In [9]:
# =========================================================
# 第二部分：特征X / 标签Y（向量化）
# =========================================================
def window_features(raw):
    """raw: dict of (n_days, n_stocks) -> X (T, S, W, 7), 其中 T = n_days-W+1，窗口末日 = i+W-1"""
    outs = []
    close_last = None
    wins = {c: sliding_window_view(raw[c], WINDOW, axis=0) for c in FEATURES}  # (T,S,W)
    base = {"price": wins["close"][:, :, -1], "volume": wins["volume"][:, :, -1],
            "amount": wins["amount"][:, :, -1]}
    with np.errstate(divide="ignore", invalid="ignore"):
        for c in FEATURES:
            b = base["volume"] if c == "volume" else base["amount"] if c == "amount" else base["price"]
            outs.append((wins[c] / b[:, :, None]).astype(np.float32))
    return np.stack(outs, axis=-1)


def build_samples():
    vw = RAW["vwap"]
    label = np.full((n_days, n_stocks), np.nan)
    label[: n_days - HORIZON - 1] = vw[HORIZON + 1:] / vw[1: n_days - HORIZON] - 1   # t+1 起算未来5日VWAP收益

    Xall = window_features(RAW)                       # (T,S,W,7)
    t_end = np.arange(WINDOW - 1, n_days)
    ok = ~np.isnan(Xall).any(axis=(2, 3)) & ~np.isnan(label[t_end]) & listed_ok[t_end]
    ti, si = np.where(ok)
    X = Xall[ti, si]
    Y = label[t_end[ti], si].astype(np.float32)
    return X, Y, t_end[ti], si


print("=" * 60, "\n第二步：构建特征与标签\n", "=" * 60)
X, Y, day_idx, stock_idx = build_samples()
print(f"样本数 {len(X)}，形状 {X.shape[1:]}，标签均值 {Y.mean():.4%}，标准差 {Y.std():.4%}")

# =========================================================
# 第三部分：样本隔离（防前视）
# =========================================================
udays = np.unique(day_idx)
n_test = int(len(udays) * 0.15)
test_days = udays[-n_test:]
inner = udays[:-n_test]
sp = int(len(inner) * 0.9)
tr_all, va_all = inner[:sp], inner[sp:]
gap = (HORIZON - 1) // 2
train_days = tr_all[: len(tr_all) - (HORIZON + 1) - gap] if len(tr_all) > HORIZON + 1 + gap else tr_all
val_days = va_all[gap:]
train_mask, val_mask, test_mask = (np.isin(day_idx, d) for d in (train_days, val_days, test_days))
print(f"训练{len(train_days)}天/{train_mask.sum()}样本，验证{len(val_days)}天/{val_mask.sum()}样本，"
      f"测试{len(test_days)}天/{test_mask.sum()}样本")

day_groups = {d: np.where(day_idx == d)[0] for d in udays}   # 预先分组，避免训练中重复扫描


# =========================================================
# 第四部分：模型与训练
# =========================================================
class GRUExpert(nn.Module):
    def __init__(self, n_features=7, hidden=16, layers=2, dropout=0.1):
        super().__init__()
        self.gru = nn.GRU(n_features, hidden, layers, batch_first=True)
        self.fc1, self.act, self.drop, self.fc2 = nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden, 1)

    def forward(self, x):
        o, _ = self.gru(x)
        return self.fc2(self.drop(self.act(self.fc1(o[:, -1, :])))).squeeze(-1)


def pearson_ic(p, t, eps=1e-8):
    p, t = p - p.mean(), t - t.mean()
    return (p * t).sum() / (torch.sqrt((p ** 2).sum() * (t ** 2).sum()) + eps)


def loss_fn(kind, p, y):
    return {"mse": lambda: nn.functional.mse_loss(p, y), "mae": lambda: nn.functional.l1_loss(p, y)}.get(
        kind, lambda: -pearson_ic(p, y))()


def train_model(loss_type="ic", max_epoch=None, patience=None):
    max_epoch, patience = max_epoch or CFG["MAX_EPOCH"], patience or CFG["PATIENCE"]
    model = GRUExpert(hidden=CFG["HIDDEN"], layers=CFG["LAYERS"])
    opt = torch.optim.Adam(model.parameters(), lr=CFG["LR"])
    Xt, Yt = torch.from_numpy(X), torch.from_numpy(Y)
    tr_days = [d for d in train_days if len(day_groups[d]) >= 5]
    vi = np.where(val_mask)[0]
    best, best_state, bad = np.inf, None, 0
    hist = dict(train_loss=[], val_loss=[], train_ic=[], val_panel_ic=[])
    for ep in range(max_epoch):
        model.train()
        ls, ics = [], []
        for d in np.random.permutation(tr_days):          # 每个batch = 单个交易日截面
            idx = day_groups[d]
            p = model(Xt[idx]); y = Yt[idx]
            loss = loss_fn(loss_type, p, y)
            opt.zero_grad(); loss.backward(); opt.step()
            ls.append(loss.item()); ics.append(pearson_ic(p.detach(), y).item())
        model.eval()
        with torch.no_grad():
            pv = model(Xt[vi]); vl = loss_fn(loss_type, pv, Yt[vi]).item(); vic = pearson_ic(pv, Yt[vi]).item()
        for k, v in zip(hist, [np.mean(ls), vl, np.mean(ics), vic]):
            hist[k].append(v)
        if vl < best - 1e-6:
            best, bad = vl, 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
        if ep == 0 or (ep + 1) % 5 == 0:
            print(f"  [{loss_type}] ep{ep+1:3d} train={hist['train_loss'][-1]:.5f} val={vl:.5f} "
                  f"trainIC={hist['train_ic'][-1]:.4f} valPanelIC={vic:.4f}")
        if bad >= patience:
            print(f"  早停于第{ep+1}轮"); break
    model.load_state_dict(best_state)
    return model, hist


print("=" * 60, "\n第四步：MSE vs IC 损失对比\n", "=" * 60)
_, hist_mse = train_model("mse", max_epoch=8, patience=4)
model_ic, hist_ic = train_model("ic")


@torch.no_grad()
def predict(model, Xn, bs=20000):
    model.eval()
    return np.concatenate([model(torch.from_numpy(Xn[i:i + bs])).numpy() for i in range(0, len(Xn), bs)])


# =========================================================
# 第五部分：样本外IC
# =========================================================
print("=" * 60, "\n第五步：样本外 IC\n", "=" * 60)
dt_ = pd.DataFrame({"day": day_idx[test_mask], "stock": stock_idx[test_mask],
                    "pred": predict(model_ic, X[test_mask]), "y": Y[test_mask]})
cs_ic = dt_.groupby("day").apply(lambda g: g["pred"].corr(g["y"])).dropna()
ts_ic = dt_.groupby("stock").apply(lambda g: g["pred"].corr(g["y"]) if len(g) >= 10 else np.nan).dropna()
panel_ic = dt_["pred"].corr(dt_["y"])
summary = {"截面IC均值": cs_ic.mean(), "截面ICIR": cs_ic.mean() / cs_ic.std(),
           "时序IC均值": ts_ic.mean(), "时序ICIR": ts_ic.mean() / ts_ic.std(), "Panel IC": panel_ic}
for k, v in summary.items():
    print(f"{k}: {v:.4f}")
pd.Series(summary).to_csv(f"{CFG['OUT_DIR']}/IC统计.csv", encoding="utf-8-sig")


# =========================================================
# 第六部分：多头（路径合并）回测
# =========================================================
def long_backtest(df, top_pct=0.1, cost=0.001):
    days = np.sort(df["day"].unique())
    r = {}
    for i, (d, g) in enumerate(df.groupby("day")):
        k = max(1, int(len(g) * top_pct))
        g = g.sort_values("pred")
        r[i] = (g["y"].iloc[-k:].mean(), g["y"].mean())     # 多头、全体等权基准
    paths, bench = [], []
    for s in range(HORIZON):
        idx = [i for i in range(s, len(days), HORIZON) if i in r]
        if len(idx) > 3:
            paths.append(np.cumprod(1 + np.array([r[i][0] - cost for i in idx])))
            bench.append(np.cumprod(1 + np.array([r[i][1] for i in idx])))
    m = min(map(len, paths))
    return np.mean([p[:m] for p in paths], 0), np.mean([b[:m] for b in bench], 0)


long_nv, bench_nv = long_backtest(dt_)
n_p = len(long_nv)
ann = lambda nv: nv[-1] ** (252 / HORIZON / n_p) - 1
print(f"多头年化≈{ann(long_nv):.2%}，等权基准年化≈{ann(bench_nv):.2%}，超额≈{ann(long_nv)-ann(bench_nv):.2%}")
plt.figure(figsize=(8, 4.5))
plt.plot(long_nv, label="多头(路径合并)"); plt.plot(bench_nv, label="股票池等权")
plt.plot(long_nv / bench_nv, label="多头超额"); plt.axhline(1, ls="--", c="gray")
plt.legend(); plt.title("截面选股多头净值（样本外）"); plt.tight_layout()
plt.savefig(f"{CFG['OUT_DIR']}/long_portfolio.png", dpi=130); plt.close()


第二步：构建特征与标签
样本数 185226，形状 (40, 7)，标签均值 0.3584%，标准差 5.9714%
训练474天/138958样本，验证52天/15385样本，测试94天/27946样本
第四步：MSE vs IC 损失对比
  [mse] ep  1 train=0.00357 val=0.00352 trainIC=-0.0008 valPanelIC=-0.0087
  [mse] ep  5 train=0.00353 val=0.00361 trainIC=-0.0001 valPanelIC=0.0000
  早停于第8轮
  [ic] ep  1 train=-0.00187 val=0.01824 trainIC=0.0019 valPanelIC=-0.0182
  [ic] ep  5 train=0.00116 val=0.02164 trainIC=-0.0012 valPanelIC=-0.0216
  [ic] ep 10 train=-0.00573 val=-0.04842 trainIC=0.0057 valPanelIC=0.0484
  [ic] ep 15 train=-0.02107 val=-0.05370 trainIC=0.0211 valPanelIC=0.0537
  [ic] ep 20 train=nan val=nan trainIC=nan valPanelIC=nan
  早停于第20轮
第五步：样本外 IC
截面IC均值: -0.0070
截面ICIR: -0.0617
时序IC均值: -0.0321
时序ICIR: -0.1927
Panel IC: 0.0089
多头年化≈-19.98%，等权基准年化≈-17.79%，超额≈-2.19%


In [10]:
# =========================================================
# 第七部分：指数择时
#   A) 个股等权合成指数（真实数据衍生）
#   B) 直接推理：中证全指真实指数行情
#      聚宽指数行情无 avg(VWAP)，用典型价 (H+L+C)/3 作为 VWAP 近似；
#      volume/money 为聚宽真实指数成交量/额。该近似会引入一定误差，需知悉。
# =========================================================
def raw_from_index(code):
    d = get_price(code, start_date=CFG["START"], end_date=CFG["END"], frequency="daily",
                  fields=["open", "close", "high", "low", "volume", "money"], skip_paused=False, fq="pre", panel=False)
    d = d.drop(columns=[c for c in ["code"] if c in d.columns])
    if "time" in d.columns:
        d = d.set_index("time")
    d.index = pd.to_datetime(d.index)
    d = d.reindex(dates)
    return {"open": d["open"].values[:, None], "close": d["close"].values[:, None],
            "high": d["high"].values[:, None], "low": d["low"].values[:, None],
            "vwap": ((d["high"] + d["low"] + d["close"]) / 3).values[:, None],   # 近似
            "volume": d["volume"].values[:, None], "amount": d["money"].values[:, None]}


def synthetic_raw(weights=None):
    w = np.ones(n_stocks) / n_stocks if weights is None else weights
    return {c: (np.nan_to_num(RAW[c]) * w).sum(1)[:, None] for c in FEATURES}


def factor_series(model, raw):
    Xw = window_features(raw)[:, 0]                      # (T,W,7)
    f = np.full(n_days, np.nan)
    ok = ~np.isnan(Xw).any(axis=(1, 2))
    f[WINDOW - 1:][ok] = predict(model, Xw[ok])
    return f, raw["close"][:, 0]


def timing_signals(f, N=40):
    s = pd.Series(f)
    ma, ma_s, sd = s.rolling(N).mean(), s.rolling(max(2, N // 4)).mean(), s.rolling(N).std()
    s1 = np.where(s > ma, 1, -1)
    s2 = np.where(ma_s > ma, 1, -1)
    s3 = np.where(s > ma + .5 * sd, 1, np.where(s < ma - .5 * sd, -1, 0))
    return s1, s2, s3


def backtest_timing(sig, close, lo=2.0, sh=0.5, ne=1.0):
    ret = np.nan_to_num(pd.Series(close).pct_change().values)
    pos = np.roll(np.where(sig == 1, lo, np.where(sig == -1, sh, ne)), 1); pos[0] = ne
    nv, b = np.cumprod(1 + pos * ret), np.cumprod(1 + ret)
    return nv, b


def eval_timing(name, fac, close, start, N=40):
    res = {}
    for nm, sg in zip(["策略1_短趋势", "策略2_长趋势", "策略3_区间突破"], timing_signals(fac, N)):
        nv, b = backtest_timing(sg[start:], close[start:])
        a_s, a_b = nv[-1] ** (252 / len(nv)) - 1, b[-1] ** (252 / len(b)) - 1
        mdd = (nv / np.maximum.accumulate(nv) - 1).min()
        res[nm] = dict(nv=nv, bench=b, ann=a_s, ann_b=a_b, mdd=mdd)
        print(f"[{name}] {nm}: 年化 {a_s:.2%} 基准 {a_b:.2%} 超额≈{a_s-a_b:.2%} 最大回撤 {mdd:.2%}")
    return res


print("=" * 60, "\n第七步：指数择时\n", "=" * 60)
start = max(int(test_days.min()), WINDOW + 60)
fac_syn, close_syn = factor_series(model_ic, synthetic_raw())
res_syn = eval_timing("个股合成指数", fac_syn, close_syn, start)

idx_raw = raw_from_index(CFG["TIMING_INDEX"])
fac_idx, close_idx = factor_series(model_ic, idx_raw)
res_idx = eval_timing("中证全指直接推理", fac_idx, close_idx, start)

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for ax_, (t, res) in zip(ax, [("个股合成指数", res_syn), ("中证全指直接推理", res_idx)]):
    for k, r in res.items():
        ax_.plot(r["nv"], label=k)
    ax_.plot(res["策略1_短趋势"]["bench"], "k", label="基准"); ax_.set_title(t); ax_.legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{CFG['OUT_DIR']}/timing.png", dpi=130); plt.close()

# =========================================================
# 第八部分：参数敏感性（N=10~80，中证全指直接推理，策略1）
# =========================================================
print("=" * 60, "\n第八步：参数敏感性\n", "=" * 60)
rows = []
for N in range(10, 90, 10):
    s1 = timing_signals(fac_idx, N)[0]
    nv, b = backtest_timing(s1[start:], close_idx[start:])
    rows.append((N, (nv[-1] ** (252 / len(nv)) - 1) - (b[-1] ** (252 / len(b)) - 1)))
    print(f"N={N}: 年化超额≈{rows[-1][1]:.2%}")
pd.DataFrame(rows, columns=["N", "年化超额"]).to_csv(f"{CFG['OUT_DIR']}/参数敏感性.csv", index=False, encoding="utf-8-sig")

# =========================================================
# 第九部分：标的敏感性（沪深300/中证800/中证1000/创业板指 真实指数）
# =========================================================
print("=" * 60, "\n第九步：标的敏感性\n", "=" * 60)
rows = []
for nm, code in [("沪深300", "000300.XSHG"), ("中证800", "000906.XSHG"),
                 ("中证1000", "000852.XSHG"), ("创业板指", "399006.XSHE")]:
    try:
        f, c = factor_series(model_ic, raw_from_index(code))
        s1 = timing_signals(f, 40)[0]
        nv, b = backtest_timing(s1[start:], c[start:])
        ex = (nv[-1] ** (252 / len(nv)) - 1) - (b[-1] ** (252 / len(b)) - 1)
        rows.append((nm, ex)); print(f"{nm}: 年化超额≈{ex:.2%}")
    except Exception as e:
        print(f"{nm} 获取失败: {e}")
pd.DataFrame(rows, columns=["指数", "年化超额"]).to_csv(f"{CFG['OUT_DIR']}/标的敏感性.csv", index=False, encoding="utf-8-sig")

# =========================================================
# 第十部分：损失函数对比图
# =========================================================
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
ax[0].plot(hist_mse["train_loss"], label="训练MSE"); ax[0].plot(hist_mse["val_loss"], label="验证MSE"); ax[0].legend()
ax[1].plot(hist_ic["train_ic"], label="训练截面IC"); ax[1].plot(hist_ic["val_panel_ic"], label="验证PanelIC")
ax[1].axhline(0, ls="--", c="gray"); ax[1].legend()
plt.tight_layout(); plt.savefig(f"{CFG['OUT_DIR']}/loss_comparison.png", dpi=130); plt.close()
print("完成，结果保存在", CFG["OUT_DIR"])


第七步：指数择时
[个股合成指数] 策略1_短趋势: 年化 -30.53% 基准 -47.19% 超额≈16.66% 最大回撤 -17.55%
[个股合成指数] 策略2_长趋势: 年化 -24.40% 基准 -47.19% 超额≈22.79% 最大回撤 -19.61%
[个股合成指数] 策略3_区间突破: 年化 -26.78% 基准 -47.19% 超额≈20.41% 最大回撤 -18.92%
[中证全指直接推理] 策略1_短趋势: 年化 -46.27% 基准 -41.39% 超额≈-4.88% 最大回撤 -22.55%
[中证全指直接推理] 策略2_长趋势: 年化 -46.65% 基准 -41.39% 超额≈-5.25% 最大回撤 -23.66%
[中证全指直接推理] 策略3_区间突破: 年化 -42.03% 基准 -41.39% 超额≈-0.64% 最大回撤 -19.86%
第八步：参数敏感性
N=10: 年化超额≈-0.44%
N=20: 年化超额≈-4.11%
N=30: 年化超额≈-14.80%
N=40: 年化超额≈-4.88%
N=50: 年化超额≈4.37%
N=60: 年化超额≈12.47%
N=70: 年化超额≈18.55%
N=80: 年化超额≈10.16%
第九步：标的敏感性
沪深300: 年化超额≈-5.67%
中证800: 年化超额≈11.73%
中证1000: 年化超额≈-3.28%
创业板指: 年化超额≈-1.61%
完成，结果保存在 ./outputs
